In [ ]:
#pip install pyodbc

In [1]:
import requests
import pyodbc as odbc

In [2]:
def executar_passo_9_equipa_inteligente():
    # 1. Configura aqui a tua chave gerada no OpenWeather
    API_KEY = "fd99f8c23afc69bd7ae75a1c2ba73777"  # Substitui pela tua chave real do OpenWeather
    
    # 2. Configurações de acesso ao servidor do DETI
    server = 'deti-sql-aulas.ua.pt'
    database = 'team_09'
    username = 'team_09'
    password = 'spring'
    
    connString = f'DRIVER={{SQL Server}};SERVER={server};DATABASE={database};UID={username};PWD={password}'
    
    try:
        print("A conectar ao SQL Server da equipa...")
        conn = odbc.connect(connString, timeout=5)
        cursor = conn.cursor()
        
        # --- TRATAMENTO DINÂMICO DA COLUNA ---
        # Lemos apenas a primeira linha para inspecionar os nomes reais das colunas
        cursor.execute("SELECT TOP 1 * FROM dbo.DepLocations")
        colunas = [col[0] for col in cursor.description]
        
        # Normalmente a estrutura é: [0] número do departamento, [1] nome da localização/cidade
        coluna_cidade = colunas[1]
        print(f"-> Estrutura detetada! A vossa coluna chama-se: '{coluna_cidade}'")
        
        # Executa o SELECT usando o nome real da coluna descoberto pelo Python
        cursor.execute(f"SELECT DISTINCT [{coluna_cidade}] FROM dbo.DepLocations")
        cidades = [linha[0] for linha in cursor.fetchall()]
        conn.close()
        
        if not cidades:
            print("A tabela dbo.DepLocations está vazia.")
            return

        # Cabeçalho da tabela formatado
        print(f"\n{'Cidade':<15} | {'Temp (°C)':<10} | {'Vento (m/s)':<12} | {'Precip. (mm)':<13} | {'Condição'}")
        print("-" * 70)
        
        # 3. Consulta à API OpenWeather para cada cidade retornada
        for city in cidades:
            # Garante que espaços em branco não quebram o URL da API
            city_clean = str(city).strip()
            
            url = f"https://api.openweathermap.org/data/2.5/weather?q={city_clean}&appid={API_KEY}&units=metric&lang=pt"
            response = requests.get(url)
            
            if response.status_code == 200:
                dados = response.json()
                
                temp = dados['main']['temp']
                wind_speed = dados['wind']['speed']
                weather_desc = dados['weather'][0]['description']
                
                rain_1h = dados.get('rain', {}).get('1h', 0)
                snow_1h = dados.get('snow', {}).get('1h', 0)
                precipitation = rain_1h if rain_1h > 0 else snow_1h
                
                print(f"{city_clean:<15} | {temp:<10} | {wind_speed:<12} | {precipitation:<13} | {weather_desc.capitalize()}")
            else:
                print(f"{city_clean:<15} | Erro na API OpenWeather (Código {response.status_code})")
                
    except odbc.Error as e:
        print("\n[ERRO DE EXECUÇÃO] Falha na comunicação com o SQL Server:")
        print(e.args[1])

if __name__ == "__main__":
    executar_passo_9_equipa_inteligente()

A conectar ao SQL Server da equipa...
-> Estrutura detetada! A vossa coluna chama-se: 'dLocation'

Cidade          | Temp (°C)  | Vento (m/s)  | Precip. (mm)  | Condição
----------------------------------------------------------------------
Aveiro          | 20.84      | 5.29         | 0             | Nublado
Lisboa          | 24.63      | 7.2          | 0             | Céu pouco nublado
Porto           | 17.73      | 4.12         | 0             | Nuvens quebradas
